In [ ]:
# Imports
import os
import gc
import glob
import shutil
import hashlib
import uuid
from datetime import datetime, timezone

import py7zr
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

print("Ambiente PySpark inicializado.")

In [ ]:
# Configuração
LAKEHOUSE_FILES_PATH = "/lakehouse/default/Files"
LAKEHOUSE_TABLES_PATH = "/lakehouse/default/Tables"

CAGED_SOURCE_DIR = os.path.join(LAKEHOUSE_FILES_PATH, "CAGED")

BRONZE_SCHEMA = "BRONZE"
BRONZE_TABLE_NAME = "CAGED"
BRONZE_TABLE_IDENTIFIER = f"{BRONZE_SCHEMA}.{BRONZE_TABLE_NAME}"

CONTROL_TABLE_NAME = "controle_caged"
CONTROL_TABLE_IDENTIFIER = f"{BRONZE_SCHEMA}.{CONTROL_TABLE_NAME}"
CONTROL_TABLE_PATH = os.path.join(LAKEHOUSE_TABLES_PATH, BRONZE_SCHEMA, CONTROL_TABLE_NAME)

CSV_DELIMITER = ";"
CSV_ENCODING = "UTF-8"
TEMP_EXTRACT_DIR = "/lakehouse/default/Files/tmp/caged_extract"

print("CAGED_SOURCE_DIR:", CAGED_SOURCE_DIR)
print("BRONZE_TABLE:", BRONZE_TABLE_IDENTIFIER)
print("CONTROL_TABLE:", CONTROL_TABLE_IDENTIFIER)

In [ ]:
# Preparação dos objetos Delta

def ensure_schema_exists() -> None:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")


CONTROL_SCHEMA = StructType([
    StructField("execution_id", StringType(), False),
    StructField("archive_name", StringType(), True),
    StructField("archive_hash", StringType(), True),
    StructField("source_file", StringType(), True),
    StructField("source_file_hash", StringType(), True),
    StructField("status", StringType(), False),
    StructField("rows_loaded", LongType(), False),
    StructField("error_count", LongType(), False),
    StructField("error_message", StringType(), True),
    StructField("started_at", TimestampType(), False),
    StructField("finished_at", TimestampType(), True),
])


def control_table_exists() -> bool:
    return spark.catalog.tableExists(CONTROL_TABLE_IDENTIFIER)


def bronze_table_exists() -> bool:
    return spark.catalog.tableExists(BRONZE_TABLE_IDENTIFIER)


def ensure_control_table_exists() -> None:
    if not control_table_exists():
        empty_df = spark.createDataFrame([], CONTROL_SCHEMA)
        (
            empty_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(CONTROL_TABLE_IDENTIFIER)
        )
        print(f"[CREATE] {CONTROL_TABLE_IDENTIFIER}")
    else:
        print(f"[EXISTS] {CONTROL_TABLE_IDENTIFIER}")


def initialize_environment() -> None:
    ensure_schema_exists()
    ensure_control_table_exists()

    if bronze_table_exists():
        print(f"[EXISTS] {BRONZE_TABLE_IDENTIFIER}")
    else:
        print(f"[INFO] {BRONZE_TABLE_IDENTIFIER} será criada no primeiro append.")


initialize_environment()

In [ ]:
# Utilitários de arquivos

def compute_file_hash(filepath: str, block_size: int = 8 * 1024 * 1024) -> str:
    sha = hashlib.sha256()
    with open(filepath, "rb") as file:
        for block in iter(lambda: file.read(block_size), b""):
            sha.update(block)
    return sha.hexdigest()


def list_7z_files() -> list[str]:
    return sorted(glob.glob(os.path.join(CAGED_SOURCE_DIR, "*.7z")))


def extract_7z(archive_path: str, target_dir: str) -> list[str]:
    shutil.rmtree(target_dir, ignore_errors=True)
    os.makedirs(target_dir, exist_ok=True)

    with py7zr.SevenZipFile(archive_path, mode="r") as archive:
        archive.extractall(path=target_dir)

    extracted_files = []
    for root, _, files in os.walk(target_dir):
        for filename in files:
            extracted_files.append(os.path.join(root, filename))

    return sorted(extracted_files)

def get_spark_path(filepath: str) -> str:
    prefix = "/lakehouse/default/"
    
    if filepath.startswith(prefix):
        return filepath[len(prefix):]
    
    return filepath

def list_data_files(extracted_files: list[str]) -> list[str]:
    return [
        path for path in extracted_files
        if os.path.isfile(path)
        and not os.path.basename(path).startswith(".")
        and path.lower().endswith(".txt")
    ]

In [ ]:
# Controle / auditoria

def get_successful_file_hashes() -> set[str]:
    if not control_table_exists():
        return set()

    rows = (
        spark.table(CONTROL_TABLE_IDENTIFIER)
        .filter(F.col("status") == "SUCCESS")
        .select("source_file_hash")
        .where(F.col("source_file_hash").isNotNull())
        .distinct()
        .collect()
    )
    return {row["source_file_hash"] for row in rows}


def source_hash_exists_in_bronze(source_file_hash: str) -> bool:
    if not bronze_table_exists():
        return False

    return (
        spark.table(BRONZE_TABLE_IDENTIFIER)
        .filter(F.col("_source_file_hash") == source_file_hash)
        .limit(1)
        .count() > 0
    )


def register_control(
    execution_id: str,
    archive_name: str,
    archive_hash: str,
    source_file: str | None,
    source_file_hash: str | None,
    status: str,
    rows_loaded: int = 0,
    error_count: int = 0,
    error_message: str | None = None,
    started_at: datetime | None = None,
    finished_at: datetime | None = None,
) -> None:
    row = [{
        "execution_id": execution_id,
        "archive_name": archive_name,
        "archive_hash": archive_hash,
        "source_file": source_file,
        "source_file_hash": source_file_hash,
        "status": status,
        "rows_loaded": int(rows_loaded),
        "error_count": int(error_count),
        "error_message": error_message,
        "started_at": started_at or datetime.now(timezone.utc),
        "finished_at": finished_at or datetime.now(timezone.utc),
    }]

    (
        spark.createDataFrame(row, schema=CONTROL_SCHEMA)
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(CONTROL_TABLE_IDENTIFIER)
    )


def show_last_execution(execution_id: str) -> None:
    (
        spark.table(CONTROL_TABLE_IDENTIFIER)
        .filter(F.col("execution_id") == execution_id)
        .orderBy("started_at", "source_file")
        .show(truncate=False)
    )

In [ ]:
# Leitura e append na Bronze

def read_source_file(filepath: str):
    spark_path = get_spark_path(filepath)

    return (
        spark.read
        .option("header", "true")
        .option("sep", CSV_DELIMITER)
        .option("encoding", CSV_ENCODING)
        .option("inferSchema", "false")
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .csv(spark_path)
    )


def append_file_to_bronze(
    filepath: str,
    source_archive: str,
    source_file_hash: str,
) -> tuple[int, int]:
    df = read_source_file(filepath)

    df = (
        df
        .withColumn("_source_archive", F.lit(os.path.basename(source_archive)))
        .withColumn("_source_file", F.lit(os.path.basename(filepath)))
        .withColumn("_source_file_hash", F.lit(source_file_hash))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
    )

    rows_loaded = df.count()

    if "_corrupt_record" in df.columns:
        error_count = df.filter(F.col("_corrupt_record").isNotNull()).count()
    else:
        error_count = 0

    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(BRONZE_TABLE_IDENTIFIER)
    )

    return rows_loaded, error_count


def process_source_file(
    filepath: str,
    archive_path: str,
    archive_hash: str,
    execution_id: str,
    successful_hashes: set[str],
) -> str:
    source_file = os.path.basename(filepath)
    source_file_hash = compute_file_hash(filepath)
    started_at = datetime.now(timezone.utc)

    # Barreira 1: auditoria.
    if source_file_hash in successful_hashes:
        register_control(
            execution_id, os.path.basename(archive_path), archive_hash,
            source_file, source_file_hash, "SKIPPED",
            error_message="Arquivo já processado com sucesso anteriormente.",
            started_at=started_at,
        )
        print(f"[SKIP] {source_file}: já consta como SUCCESS.")
        return "SKIPPED"

    # Barreira 2: a própria Bronze.
    if source_hash_exists_in_bronze(source_file_hash):
        register_control(
            execution_id, os.path.basename(archive_path), archive_hash,
            source_file, source_file_hash, "SKIPPED",
            error_message="Hash do arquivo já encontrado na Bronze.",
            started_at=started_at,
        )
        print(f"[SKIP] {source_file}: hash já encontrado na Bronze.")
        return "SKIPPED"

    try:
        print(f"[PROCESS] {source_file} ...")

        rows_loaded, error_count = append_file_to_bronze(
            filepath=filepath,
            source_archive=archive_path,
            source_file_hash=source_file_hash,
        )

        register_control(
            execution_id, os.path.basename(archive_path), archive_hash,
            source_file, source_file_hash, "SUCCESS",
            rows_loaded=rows_loaded,
            error_count=error_count,
            started_at=started_at,
        )

        successful_hashes.add(source_file_hash)

        print(
            f"[OK] {source_file}: {rows_loaded:,} linhas | "
            f"{error_count:,} registros com erro de parsing."
        )
        return "SUCCESS"

    except Exception as exc:
        register_control(
            execution_id, os.path.basename(archive_path), archive_hash,
            source_file, source_file_hash, "ERROR",
            rows_loaded=0,
            error_count=1,
            error_message=str(exc),
            started_at=started_at,
        )
        print(f"[ERROR] {source_file}: {exc}")
        return "ERROR"


def process_archive(
    archive_path: str,
    execution_id: str,
    successful_hashes: set[str],
) -> None:
    archive_name = os.path.basename(archive_path)
    archive_hash = compute_file_hash(archive_path)
    extract_dir = os.path.join(TEMP_EXTRACT_DIR, archive_hash)

    print(f"\n[ARCHIVE] {archive_name}")

    try:
        extracted_files = extract_7z(archive_path, extract_dir)
        data_files = list_data_files(extracted_files)

        if not data_files:
            raise RuntimeError("Nenhum arquivo .txt encontrado após a extração.")

        for filepath in data_files:
            process_source_file(
                filepath, archive_path, archive_hash,
                execution_id, successful_hashes
            )

    except Exception as exc:
        register_control(
            execution_id, archive_name, archive_hash,
            None, None, "ERROR",
            rows_loaded=0,
            error_count=1,
            error_message=f"Falha no .7z: {exc}",
        )
        print(f"[ERROR] {archive_name}: {exc}")

    finally:
        shutil.rmtree(extract_dir, ignore_errors=True)
        gc.collect()

In [ ]:
# Execução principal

def main() -> str | None:
    initialize_environment()

    archives = list_7z_files()
    if not archives:
        print(f"Nenhum arquivo .7z encontrado em: {CAGED_SOURCE_DIR}")
        return None

    execution_id = str(uuid.uuid4())
    started_at = datetime.now(timezone.utc)
    successful_hashes = get_successful_file_hashes()

    print("=" * 80)
    print(f"EXECUÇÃO: {execution_id}")
    print(f"Arquivos encontrados: {len(archives)}")
    print(f"Arquivos já processados com sucesso: {len(successful_hashes)}")
    print("=" * 80)

    os.makedirs(TEMP_EXTRACT_DIR, exist_ok=True)

    for archive_path in archives:
        process_archive(
            archive_path,
            execution_id,
            successful_hashes,
        )

    shutil.rmtree(TEMP_EXTRACT_DIR, ignore_errors=True)

    finished_at = datetime.now(timezone.utc)

    print("=" * 80)
    print("PROCESSAMENTO CONCLUÍDO")
    print(f"Início: {started_at}")
    print(f"Fim   : {finished_at}")
    print(f"ID    : {execution_id}")
    print("=" * 80)

    show_last_execution(execution_id)
    return execution_id


execution_id = main()